In [1]:
import pandas as pd

In [2]:
frontage = "data_files/Frontpage.csv"
process = "data_files/Process.csv"

In [3]:
frontage_df = pd.read_csv(frontage)
process_df = pd.read_csv(process)


In [4]:
print(frontage_df.columns)

Index(['Item', 'SAP TN', 'SAP PL', 'DCC Type', 'Description', '2024', '2025',
       '2026', '2027', '2028', '2029', '2030'],
      dtype='object')


In [23]:
import pandas as pd

# Rename columns
df = frontage_df.rename(columns={
    'SAP TN': 'SAP_TN',
    'SAP PL': 'SAP_PL',
    'DCC Type': 'DCC_Type'
})

# Select required columns
df = df[['Item', 'SAP_TN', 'SAP_PL', 'DCC_Type', 'Description', '2024']]

# Rename year column
df = df.rename(columns={'2024': 'Demand_2024'})

# Columns to convert to integer
int_cols = ['Item', 'SAP_TN', 'SAP_PL', 'Demand_2024']

# Clean commas and convert to integer (nullable Int64 for safety)
for col in int_cols:
    df[col] = (
        df[col]
        .astype(str)                  # convert all to string first
        .str.replace(',', '', regex=False)  # remove commas
        .replace('None', pd.NA)       # convert literal 'None' to NA
        .pipe(pd.to_numeric, errors='coerce')  # convert to numeric, invalid → NaN
        .astype('Int64')               # nullable integer type
    )

# Take first 5 rows
df = df.head(23)

# Replace NaN with None for JSON/dict output
df = df.where(pd.notna(df), None)

# Convert to dictionary
demand_data = df.to_dict(orient='list')

In [24]:
display(demand_data)

{'Item': [1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23],
 'SAP_TN': [249076,
  249077,
  249313,
  249314,
  249315,
  249316,
  249317,
  249078,
  249079,
  249080,
  249081,
  249082,
  249083,
  249084,
  249085,
  249086,
  249087,
  249088,
  249089,
  249090,
  249091,
  249092,
  249093],
 'SAP_PL': [249041,
  249042,
  238895,
  238896,
  238897,
  238899,
  238900,
  249043,
  249044,
  249045,
  249046,
  249047,
  249048,
  249049,
  249050,
  249051,
  249052,
  249053,
  249054,
  249055,
  249056,
  249057,
  249058],
 'DCC_Type': ['60° & 30°',
  '60° & 30°',
  '60° & 90°B',
  '60° & 90°B',
  '60° & 2*90°B',
  '60° & 90°B',
  '60° & 2*90°B',
  '30°',
  '90°B',
  '90°',
  '60°',
  '60°',
  '60°',
  '180°',
  '180°',
  '30°',
  '90°B',
  '60°',
  '90°',
  '60°',
  '60°',
  '180°',
  '180°'],
 'Description': ['4 Wire Jacket 2xDCC Modul 9Y4252B',
  '4 Wire Jacket 2xDCC Modul 9Y4256B',
  '4 Wire 

In [ ]:
import pandas as pd
import re

process_routing = []

machines = (
    process_df.iloc[2, 4:]
    .fillna('')
    .astype(str)
    .str.strip()
    .tolist()
)

process_steps = (
    process_df.iloc[3, 4:]
    .fillna('')
    .astype(str)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .tolist()
)

data_df = process_df.iloc[4:].copy()

for _, row in data_df.iterrows():
    # Skip rows without ITEM number
    if not str(row.iloc[0]).isdigit():
        continue

    item = int(row.iloc[0])

    for idx in range(len(process_steps)):
        raw_val = row.iloc[idx + 4]

        # Convert to number safely
        time_val = pd.to_numeric(raw_val, errors='coerce')

        if pd.notna(time_val) and time_val > 0:
            process_routing.append({
                'item': item,
                'step': idx + 1,
                'machine': machines[idx],
                'time': round(float(time_val), 2),
                'name': process_steps[idx],
                'workers': 0.5
            })


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Process,1,2,3,4,5,6,...,193,194,195,196,197,198,199,200,Unnamed: 204,Unnamed: 205
4,1,249076,4 Wire Jacket 2xDCC Modul 9Y4252B,60° & 30°,NaN,NaN,NaN,6.76,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,114.06,1.9
5,2,249077,4 Wire Jacket 2xDCC Modul 9Y4256B,60° & 30°,NaN,NaN,NaN,6.76,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,114.06,1.9
6,3,249313,4 Wire Jacket 2xDCC Modul 9Y4251,60° & 90°B,7.3,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57.2,0.95
7,4,249314,6 Wire Jacket 2xDCC Modul 9Y4251A,60° & 90°B,7.29,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,53.48,0.89
8,5,249315,6 Wire Jacket 3xDCC Modul 9Y4252,60° & 2*90°B,7.19,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57.09,0.95


In [17]:
# print(machines)
machines_new =machines
machines_new = list(set(machines_new))
print(machines_new)

['', 'SKM DCPC Crimp (Crimp and Ass)', 'ARBURG 375ST Machine 6-10', 'ARBURG 375ST Machine 11,12,13', 'Kappa 350 / Kappa 330', 'Alpha 550 / Alpha 433', 'PUR-Tube Assembly Station', 'Wire Rolling & Taping Station', 'Connector Assembly Station', 'SKM Seal and outer Housing Assembly DCC', 'Cutting Automation', 'TSK T1500', 'ARBURG 375ST Machine 1-5', 'Sigma 688 / Alpha 488', 'Wire Cut & Separating Station']


In [20]:
from pprint import pprint

pprint(process_routing[:200])


[{'item': 1,
  'machine': 'Kappa 350 / Kappa 330',
  'name': 'Cutting Stripping Jacket Cable 4-Wire',
  'step': 4,
  'time': 6.76,
  'workers': 0.5},
 {'item': 1,
  'machine': 'Wire Cut & Separating Station',
  'name': 'Separating & Cutting Wires to Length 1 of 2 Pairs',
  'step': 10,
  'time': 8.12,
  'workers': 0.5},
 {'item': 1,
  'machine': 'PUR-Tube Assembly Station',
  'name': 'Assembly PUR-Tube 3,5x1,35mm Jacket Cable 111mm-200mm',
  'step': 18,
  'time': 20.88,
  'workers': 0.5},
 {'item': 1,
  'machine': 'SKM DCPC Crimp (Crimp and Ass)',
  'name': 'Crimping & Assembly DCC Connector',
  'step': 30,
  'time': 20.0,
  'workers': 0.5},
 {'item': 1,
  'machine': 'ARBURG 375ST Machine 11,12,13',
  'name': 'Overmolding 60° Right 9J1 973 752 A Cod. C Blue Cod. Up With CPA',
  'step': 57,
  'time': 18.72,
  'workers': 0.5},
 {'item': 1,
  'machine': 'ARBURG 375ST Machine 6-10',
  'name': 'Overmolding 30° Left 9Y4 973 752 Cod. A Black Cod. Up With CPA',
  'step': 59,
  'time': 20.0,
  '